# Pythia 410m

In [27]:
import sys
sys.path.insert(0, '../..')
from src.models import load_model, unload

### Load Model

In [28]:
model_name = "pythia-410m"
model, info = load_model(model_name)

Loaded pretrained model pythia-410m into HookedTransformer
Loaded pythia-410m on mps
  24 layers | 16 heads | d_model=1024 | d_mlp=4096 | parallel attn+MLP


### Baseline Prompt

In [29]:
prompt = "A screen reader is"
output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
print(f"Input: {prompt}")
print(f"Output: {output}")

# Cache internal states
logits, cache = model.run_with_cache(prompt)
print(f"\nCached {len(cache)} different activation points!")

Input: A screen reader is
Output: A screen reader is a device that allows a user to view a screen

Cached 435 different activation points!


### Declarative Prompts

In [30]:
test_prompts = [
    "A screen reader is",
    "WCAG stands for", 
    "A skip link is",
    "The purpose of alt text is",
    "ARIA stands for",
    "A focus indicator is",
    "Keyboard navigation allows",
    "Color contrast is important because",
    "Semantic HTML helps",
    "Captions are used for",
]

for prompt in test_prompts:
    output = model.generate(prompt, max_new_tokens=10, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

A screen reader is             →  a device that allows a user to view a screen
WCAG stands for                →  the World Confederation of Agricultural and Food Industries.
A skip link is                 →  a link that is not part of the main page
The purpose of alt text is     →  to provide a way for users to customize the text
ARIA stands for                →  the acronym for the acronym for
A focus indicator is           →  a device that is used to indicate the presence of
Keyboard navigation allows     →  you to navigate through the various sections of the site
Color contrast is important because →  it can be used to distinguish between different colors.
Semantic HTML helps            →  you create a rich, interactive website.


Captions are used for          →  the purpose of providing a clear and concise description of


### Evaluative Prompts

In [31]:
code_prompts = [
    "The following code is not accessible because it doesn't have what? <img src='photo.jpg'>",
    "A <div> with onclick is not accessible because",
    "The accessibility problem with <a href='#'></a> is",
    "<input type='text'> needs a",
    "A button that only says 'Click here' is bad because",
]
for prompt in code_prompts:
    output = model.generate(prompt, max_new_tokens=20, temperature=0, verbose=False)
    print(f"{prompt:30} → {output[len(prompt):]}")

The following code is not accessible because it doesn't have what? <img src='photo.jpg'> → 

The following code is not accessible because it doesn't have what? <img src='photo
A <div> with onclick is not accessible because →  it is not a valid HTML element.

A <div> with onclick is not accessible because
The accessibility problem with <a href='#'></a> is →  that it is not clear what the user is looking for.

The <a href='#'
<input type='text'> needs a    →  <input type='text'>

<input type='submit'>

</form>

A button that only says 'Click here' is bad because →  it's not a link.

A button that only says 'Click here' is bad because


### Perplexity

In [32]:
import torch

correct = "A screen reader is software that reads text aloud for blind users."
wrong = "A screen reader is a device for viewing screens."

def get_perplexity(model, text):
    tokens = model.to_tokens(text)
    logits = model(tokens)
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    
    # Get log prob of each actual next token
    token_log_probs = log_probs[0, :-1, :].gather(1, tokens[0, 1:].unsqueeze(1)).squeeze()
    
    # Perplexity = exp(-mean(log_probs))
    return torch.exp(-token_log_probs.mean()).item()

print(f"Correct: {get_perplexity(model, correct)}")
print(f"Wrong: {get_perplexity(model, wrong)}")

Correct: 40.05073165893555
Wrong: 32.803348541259766


### Attention Binding

In [33]:
import pandas as pd

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
print(list(enumerate(tokens)))  # verify indices

logits, cache = model.run_with_cache(prompt)

threshold = 0.1
rows = []

for layer in range(model.cfg.n_layers):
    attention = cache["pattern", layer]
    for head in range(model.cfg.n_heads):
        attn = attention[0, head]
        reader_idx = 3
        screen_idx = 2
        score = attn[reader_idx, screen_idx].item()
        if score > threshold:
            rows.append({
                "layer": layer,
                "head": head,
                "reader_to_screen": round(score, 4)
            })

out_path = (f"../results/{model_name}")
df = pd.DataFrame(rows)
df.to_csv(f"../results/{model_name}/attention-binding.csv", index=False)


print("\nTop 10 by binding strength:")
sorted_rows = sorted(rows, key=lambda x: x["reader_to_screen"], reverse=True)
for row in sorted_rows[:10]:
    print(f"Layer {row['layer']:2d}, Head {row['head']:2d}: {row['reader_to_screen']}")    

print(f"\nFound {len(rows)} heads above threshold")
print(f"Saved to {out_path}")

[(0, '<|endoftext|>'), (1, 'A'), (2, ' screen'), (3, ' reader'), (4, ' is')]

Top 10 by binding strength:
Layer  0, Head 14: 0.9758
Layer  3, Head 13: 0.9591
Layer  1, Head  3: 0.9404
Layer  3, Head  3: 0.8673
Layer  5, Head  2: 0.8562
Layer  3, Head  4: 0.8519
Layer  0, Head  0: 0.8454
Layer  1, Head 10: 0.8005
Layer  3, Head  5: 0.7801
Layer  2, Head 13: 0.7638

Found 71 heads above threshold
Saved to ../results/pythia-410m


In [35]:
import circuitsvis as cv

prompt = "A screen reader is"
tokens = model.to_str_tokens(prompt)
logits, cache = model.run_with_cache(prompt)

# Visualize attention for a specific layer (start with the last layer)
layer = 11
attention = cache["pattern", layer]  # shape: [heads, seq, seq]
print(f"Layer 0, Head 14")

html = cv.attention.attention_patterns(tokens=tokens, attention=attention[0])

html

Layer 0, Head 14


In [36]:
unload(model)

Model unloaded, memory cleared
